# 🏷️ 02. Container Behavioral Clustering & Profiling

Keep Zakaria’s K-Means workflow, now with one profile per location **and material**, a historical cutoff, corrected cycle rates, and explicit sparse-history labels.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT / "Zakaria-Work"
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from Zakaria-Work or its parent directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from src.data_loader import load_and_clean_data, collection_summary
from src.clustering import extract_container_profiles, cluster_containers
from src.predictor import predict_container_demands, select_collection_sites
OUT = ROOT / "data" / "corrected"
OUT.mkdir(parents=True, exist_ok=True)
MAT_COLORS = {"white": "#1f77b4", "green": "#2ca02c", "brown": "#8c564b"}
plt.rcParams.update({"figure.figsize": (13, 5), "figure.dpi": 100})


## 1. Load validated source
Recompute from the raw CSV through the shared loader. This prevents an old generated CSV with different semantics from silently entering the analysis.

In [ ]:
df = load_and_clean_data(ROOT / "glass_fill_levels_anonymized_f.csv")
print(f"{len(df):,} pre-emptying observations; {df.Location_id.nunique()} locations; "
      f"{len(df.groupby(['Location_id', 'material_type']))} observed bins")
print(f"Coverage: {df.date.min().date()} to {df.date.max().date()}")
display(df.head())


## 2. Historical features
Each dispatch profile uses only earlier observations. A cycle-rate proxy is the next pre-emptying fill divided by days since the previous emptying. Full/overfull cycles give lower bounds. Bin medians fall back to a same-material median only when eligible history exists; otherwise rate is unavailable. Numeric means exclude overflow.

In [ ]:
# Change this explicit morning dispatch date to replay a different day.
PLANNING_DATE = pd.Timestamp("2026-08-17")
profiles = extract_container_profiles(df, as_of=PLANNING_DATE, inclusive=False)
clustered_df, kmeans_model = cluster_containers(profiles, n_clusters=4)
print(f"Known bins strictly before dispatch: {len(profiles)}")


## 3. K-Means descriptive groups
Standardize numeric features and log-transform rate to reduce scale sensitivity. Up to four distinct groups are fitted with seed 42. Small/identical datasets reduce cluster count; sparse bins stay visible. Groups are ranked by observed mean fill plus overflow share, not asserted balanced or predictive operational tiers.

In [ ]:
tier_summary = clustered_df.groupby('tier').agg(Bins=('Location_id','size'), Numeric_Mean=('avg_fill_perc','mean'),
    Explicit_Overfull_Pct=('overfull_rate','mean'), Median_Cycle_Rate=('daily_fill_velocity','median'))
display(tier_summary.round(2))
print('Sampled silhouette:', clustered_df.attrs.get('silhouette'))
print('Group sizes:', clustered_df.attrs.get('cluster_sizes'))


---
## 📊 4. Cluster Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, sub in clustered_df.groupby('tier'):
    axes[0].scatter(sub.avg_fill_perc, sub.overfull_rate, label=f"{name} (n={len(sub)})", alpha=0.65)
axes[0].set(xlabel='Mean numeric pre-emptying fill (%)', ylabel='Explicit overfull observations (%)', title='Behavioral groups at the planning cutoff')
axes[0].legend(fontsize=8)
sns.boxplot(data=clustered_df, x='material_type', y='daily_fill_velocity', ax=axes[1])
axes[1].set(xlabel='Glass material', ylabel='Cycle-median rate proxy (pp/day)', title='Rates from completed filling cycles')
plt.tight_layout()
plt.show()


## 5. Save cutoff-specific profiles
The profile artifact records `profile_as_of` and whether its cutoff was inclusive. Morning prediction rejects inclusive same-day profiles.

In [ ]:
clustered_df.to_csv(OUT / 'container_clusters.csv', index=False)
print('Saved', OUT / 'container_clusters.csv')
